# Civic Engagement — When There Is No Sampling Error At All

Every source in this corpus so far is an estimate of something. It has a margin of error, or a
suppression rule, or a model, or a coverage share, or a privacy parameter — some statement,
well made or badly made, about the distance between the number and the world.

Presidential election returns have none of that, because there is no distance to state. These
are counted ballots. No estimator was applied, no sample was drawn, and **no column in the file
could carry an uncertainty figure**. Fifteen regimes in, this is the first source whose
sampling error is exactly zero.

It is not therefore a source without traps. It has four, and the sharpest one punishes
carefulness.

The file's accounting identity — candidate votes summing to the published total — holds in
**661 of 663 state-years**, adrift by 3,409 votes in all. Then you notice that `candidate` is
not always a person: 48 rows name things like `OVERVOTES`, `UNDERVOTES` and `VOID`. Removing
them is what a careful analyst does first. **Do it and the identity holds in 625 of 663, with
the discrepancy 139 times larger** — because `totalvotes` counts those rows in most states.
The cleaning step is the bug.

The other three: the same column means different things in two states in the same year;
`version` carries two vintages in two date formats inside one download; and `party_simplified`
collapses **207 distinct parties** into `OTHER`, including Ross Perot's **19,722,042** votes in
1992, which is a fifth of that year's national total sitting in a residual bucket.

## Learning Objectives

1. **Read a source that has no uncertainty statement because none exists**, and separate that
   from a source that has one and withholds it.
2. **Test an accounting identity before and after cleaning**, and treat a worsening as evidence
   about your assumption rather than about the data.
3. **Recognise a column whose meaning is not constant across units**, and stop before
   aggregating it.
4. **Detect two vintages inside one download**, and know why a lexical sort of a version column
   is not a chronological one.
5. **Distinguish a lossy recode that is safe for the headline from one that is fatal for the
   field it collapses.**
6. **Pin a bulk file against the publisher's own checksum**, not only against your own.

## Prerequisites

- Handbook §1.11 (BEA ACPSA) — the nearest relative: a source with no sampling error because
  nothing was sampled. There the residual risk is the definitional boundary; here it is the
  boundary *and* an identity that does not close.
- Handbook §3 (denominators, universes and partitions).
- Handbook §5.3 (vintage pinning) — this notebook uses a stronger pin than D27's and section 1
  says why.

## Data Sources

**MIT Election Data and Science Lab, *U.S. President 1976–2024*,** Harvard Dataverse
`doi:10.7910/DVN/42MVDX`, file `1976-2024-president.csv`.

No API key, no registration, no terms of use, `restricted: false`. The corpus's source survey
recorded this as needing registration; it does not, and that claim has been corrected.

**Why Dataverse is a better bulk source than most.** It serves a machine-readable manifest
carrying a dataset version, a release time, and a **per-file MD5 that the publisher computed**.
A self-computed hash — the Zillow pin in D27 — proves your copy has not changed. A publisher
checksum proves the file has not been *swapped*. The client verifies against the publisher's
value on every call, and `expected_md5` additionally pins a vintage so a deliberate upstream
revision raises rather than silently rewriting every number below.

Two client behaviours worth stating because they were measured, not assumed:

- **The file is resolved by name, not by id.** Dataverse file ids change when a dataset is
  revised, so an id-addressed client pins itself to a vintage and then 404s.
- **Requests carry an identifying user agent.** This host answers HTTP 403 to the default
  `python-requests` agent and 200 to a named one. Identifying yourself to a public archive is
  the right behaviour rather than a workaround, and the client raises a message naming the
  cause if it ever fires.

## 1. Setup

The MD5 below is the one Harvard Dataverse published for dataset v10.1. It is passed on every
run. When it fires, something upstream changed — read what, then move the pin deliberately.

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    ELECTION_BALLOT_DISPOSITIONS,
    ELECTION_WRITE_IN_LABEL,
    MIT_PRESIDENT_DOI,
    MIT_PRESIDENT_FILE,
    fetch_mit_president,
)

#: The MD5 Harvard Dataverse published for this file at dataset v10.1. Passing it
#: turns any upstream revision into a loud failure instead of a silent change in
#: every number below. When it fires, read what changed and move the pin
#: deliberately -- do not delete it.
PINNED_MD5 = "405af83db7625cb35d8c19a5ebe029ff"

#: The accounting identity this notebook tests, per state and election year:
#: the candidate votes should sum to the published total.
IDENTITY = "sum(candidatevotes) == totalvotes, within each state-year"

pd.set_option("display.width", 112)
print(f"dataset   : {MIT_PRESIDENT_DOI}")
print(f"file      : {MIT_PRESIDENT_FILE}")
print(f"pinned md5: {PINNED_MD5}")
print(f"identity  : {IDENTITY}")
print(f"flagged as ballot dispositions: {', '.join(sorted(ELECTION_BALLOT_DISPOSITIONS))}")
print(f"deliberately NOT flagged      : {ELECTION_WRITE_IN_LABEL}")


dataset   : doi:10.7910/DVN/42MVDX
file      : 1976-2024-president.csv
pinned md5: 405af83db7625cb35d8c19a5ebe029ff
identity  : sum(candidatevotes) == totalvotes, within each state-year
flagged as ballot dispositions: BLANK, NONE OF THESE CANDIDATES, OVERVOTES, SCATTERING, UNDERVOTES, VOID
deliberately NOT flagged      : WRITE-IN


In [2]:
"""Retrieve, verified against the publisher's own checksum."""
returns, prov = fetch_mit_president(expected_md5=PINNED_MD5)

print(f"rows            : {len(returns):,}")
print(f"election years  : {returns['year'].nunique()} ({returns['year'].min()}-{returns['year'].max()})")
print(f"states          : {returns['state'].nunique()}")
print(f"state-years     : {returns.groupby(['year', 'state']).ngroups}")
print(f"vintage         : {prov.vintage}")
print(f"payload sha256  : {prov.content_sha256[:16]}...")
print()
print("what this source is:")
for note in prov.notes[3:5]:
    print(f"  - {note}")


rows            : 4,822
election years  : 13 (1976-2024)
states          : 51
state-years     : 663
vintage         : dataset v10.1, released 2026-09-04T16:15:49Z
payload sha256  : 9936410a6dd4ecc8...

what this source is:
  - ADMINISTRATIVE ENUMERATION. These are counted ballots, not a sample, so no margin of error exists anywhere in the file. The residual uncertainty is definitional, and it is not small.
  - `candidate` is not always a person: 48 of 4,822 rows name a ballot disposition (BLANK, NONE OF THESE CANDIDATES, OVERVOTES, SCATTERING, UNDERVOTES, VOID), carrying 473,830 votes. They are FLAGGED and NOT removed: `totalvotes` counts them in most states, so dropping them breaks the accounting identity rather than restoring it.


## 2. A file with nothing to be uncertain about

Before anything else, establish the claim the rest of the notebook depends on.

In [3]:
"""There is no margin of error anywhere in this file, and that is not reassurance."""
moe_like = [c for c in returns.columns
            if any(token in c.lower() for token in ("moe", "_m", "error", "se", "ci"))]
print(f"columns: {list(returns.columns)}")
print()
print(f"columns that could carry an uncertainty statement: {moe_like or 'none'}")
print()
print("These are counted ballots, not a sample of them. No estimator was applied,")
print("so there is no sampling error to publish and none is published.")
print()
print("Every other source in this corpus states its uncertainty somewhere, well or")
print("badly. This one has nothing to state. The rest of this notebook is about")
print("what is left over when sampling error is exactly zero.")


columns: ['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic', 'office', 'candidate', 'party_detailed', 'writein', 'candidatevotes', 'totalvotes', 'version', 'notes', 'party_simplified', 'is_ballot_disposition']

columns that could carry an uncertainty statement: none

These are counted ballots, not a sample of them. No estimator was applied,
so there is no sampling error to publish and none is published.

Every other source in this corpus states its uncertainty somewhere, well or
badly. This one has nothing to state. The rest of this notebook is about
what is left over when sampling error is exactly zero.


**Sixteen columns and not one of them could hold a margin of error.** The file
carries year, state, five state identifiers, office, candidate, two party columns, a write-in
flag, two vote counts, a version, an all-null notes column, and the flag this client adds.

That is the correct design. These are counted ballots. There is no estimator, no sample, and
therefore no sampling error to publish — so the absence of an uncertainty column is not an
omission, the way BLS LAUS's absence is (handbook §1.2). Nothing was withheld. Nothing exists.

What that buys is narrower than it sounds, and the rest of this notebook is about the
difference.

## 3. Does the arithmetic close?

The file publishes both the parts and the whole: `candidatevotes` per row and `totalvotes` per
state-year. That is an identity you can check, which most sources do not offer at all.

In [4]:
"""Does the arithmetic close, as published?"""
def balance(frame: pd.DataFrame) -> pd.DataFrame:
    grouped = frame.groupby(["year", "state"]).agg(
        summed=("candidatevotes", "sum"),
        published=("totalvotes", "first"),
        distinct_totals=("totalvotes", "nunique"),
    )
    grouped["difference"] = grouped["summed"] - grouped["published"]
    return grouped

published = balance(returns)
assert (published["distinct_totals"] == 1).all(), (
    "totalvotes must be constant within a state-year before it can be compared to a sum"
)
exact = int((published["difference"] == 0).sum())
print(f"state-years where the sum equals the published total: {exact} of {len(published)}")
print(f"total absolute discrepancy: {int(published['difference'].abs().sum()):,} votes")
print()
print(published[published["difference"] != 0][["summed", "published", "difference"]].to_string())


state-years where the sum equals the published total: 661 of 663
total absolute discrepancy: 3,409 votes

                            summed  published  difference
year state                                               
2024 DISTRICT OF COLUMBIA   328404     325869        2535
     NEW YORK              8380555    8381429        -874


**661 of 663 state-years balance exactly**, and the whole file is adrift by
**3,409 votes** out of roughly 1.5 billion cast. For a hand-assembled compilation of fifty-one
jurisdictions across thirteen elections, that is a remarkable standard.

The two exceptions are both in 2024. The District of Columbia's parts exceed its published
total by **2,535**; New York's fall short by **874**. Section 5 takes them one at a time,
because they fail for different reasons and the difference is the point.

## 4. The cleaning step that breaks it

Look at the `candidate` column and something is wrong with it. Alongside people it carries
`OVERVOTES`, `UNDERVOTES`, `VOID`, `SCATTERING` and `NONE OF THESE CANDIDATES` — ballot
dispositions, not candidacies. Grouping by candidate would mix them with human beings.

The obvious response is to drop them. Test what that does to the identity.

In [5]:
"""Now remove the rows that are obviously not people, and test again."""
flagged = returns[returns["is_ballot_disposition"]]
print(f"rows flagged as a ballot disposition: {len(flagged)} of {len(returns):,}")
print(f"votes on those rows                 : {int(flagged['candidatevotes'].sum()):,}")
print("labels present in this vintage      : "
      f"{', '.join(sorted(flagged['candidate'].unique()))}")
print()

cleaned = balance(returns[~returns["is_ballot_disposition"]])
exact_cleaned = int((cleaned["difference"] == 0).sum())
print(f"as published        : {exact} of {len(published)} balance, "
      f"{int(published['difference'].abs().sum()):,} votes adrift")
print(f"dispositions removed: {exact_cleaned} of {len(cleaned)} balance, "
      f"{int(cleaned['difference'].abs().sum()):,} votes adrift")
print()
print(f"removing them turns {len(published) - exact} failures into "
      f"{len(cleaned) - exact_cleaned}, and multiplies the discrepancy by "
      f"{cleaned['difference'].abs().sum() / published['difference'].abs().sum():.0f}x.")
print()
print("worst five after the removal:")
worst = cleaned[cleaned["difference"] != 0].reindex(
    cleaned[cleaned["difference"] != 0]["difference"].abs().sort_values(ascending=False).index
)
print(worst[["summed", "published", "difference"]].head(5).to_string())


rows flagged as a ballot disposition: 48 of 4,822
votes on those rows                 : 473,830
labels present in this vintage      : NONE OF THESE CANDIDATES, OVERVOTES, SCATTERING, UNDERVOTES, VOID

as published        : 661 of 663 balance, 3,409 votes adrift
dispositions removed: 625 of 663 balance, 472,169 votes adrift

removing them turns 2 failures into 38, and multiplies the discrepancy by 139x.

worst five after the removal:
                      summed  published  difference
year state                                         
2024 NEW YORK        8262500    8381429     -118929
2016 NEW YORK        7753637    7802084      -48447
     NORTH CAROLINA  4694178    4741564      -47386
2024 MASSACHUSETTS   3473668    3512930      -39262
2016 COLORADO        2748735    2780220      -31485


**Removing them turns 2 failures into 38 and multiplies the discrepancy by
139×** — from 3,409 votes to **472,169**.

The reason is simple once seen and invisible before: **`totalvotes` counts those rows.** The
publisher's total is ballots, not votes for people. Dropping the disposition rows removes them
from the numerator while leaving them in the denominator, and the identity that held now fails
in thirty-six more places.

This is the most useful thing in the notebook, and it generalises past elections. The rows
*are* anomalous. Removing them *is* defensible. And the operation is still wrong, because the
published total was computed over the population you just changed.

**[rule]** When a source publishes both parts and a whole, test the identity **before** you
clean and **after**. If cleaning makes it worse, your assumption about what the whole counts is
wrong — not the data.

The client therefore flags these rows and refuses to remove them. A test pins the refusal, so
a future tidy-up has to argue with it.

## 5. The same column, two meanings, one year

Both 2024 failures involve disposition rows. They do not fail the same way.

In [6]:
"""The same column, two states, one year, two meanings."""
for state in ("DISTRICT OF COLUMBIA", "NEW YORK"):
    rows = returns[(returns["year"] == 2024) & (returns["state"] == state)]
    disposition_votes = int(rows.loc[rows["is_ballot_disposition"], "candidatevotes"].sum())
    gap = int(rows["candidatevotes"].sum() - rows["totalvotes"].iloc[0])
    print(f"{state} 2024")
    print(f"  published totalvotes         : {int(rows['totalvotes'].iloc[0]):,}")
    print(f"  summed candidatevotes        : {int(rows['candidatevotes'].sum()):,}")
    print(f"  difference                   : {gap:+,}")
    print(f"  votes on disposition rows    : {disposition_votes:,} "
          f"({', '.join(sorted(rows.loc[rows['is_ballot_disposition'], 'candidate']))})")
    print(f"  difference equals them exactly: {gap == disposition_votes}")
    print()


DISTRICT OF COLUMBIA 2024
  published totalvotes         : 325,869
  summed candidatevotes        : 328,404
  difference                   : +2,535
  votes on disposition rows    : 2,535 (OVERVOTES, UNDERVOTES)
  difference equals them exactly: True

NEW YORK 2024
  published totalvotes         : 8,381,429
  summed candidatevotes        : 8,380,555
  difference                   : -874
  votes on disposition rows    : 118,055 (SCATTERING, UNDERVOTES, VOID)
  difference equals them exactly: False



**In the District of Columbia the discrepancy is exactly the disposition rows.**
`OVERVOTES` plus `UNDERVOTES` is 2,535, and the sum exceeds the published total by 2,535 —
to the vote. So DC's `totalvotes` *excludes* them while listing them as rows.

**In New York it is not.** Its disposition rows carry far more than its 874-vote gap, and
removing them leaves the state 118,929 short rather than balanced. Whatever New York's
`totalvotes` counts, it is neither "votes for people" nor "all rows listed".

So `totalvotes` does not have one definition in this file. It has at least two, in the same
year, in adjacent rows, with nothing marking which is which. That is the residual uncertainty
of an administrative enumeration: not error in the counting, but **ambiguity in what was
counted**, unstated and undetectable except by testing an identity.

**[rule]** A column whose meaning varies by unit cannot be aggregated across units, however
precise each value is. Precision is not a defence against a definition that moves.

## 6. One download, two vintages

A file retrieved in one call at one version is not necessarily one vintage of data.

In [7]:
"""One download, two vintages, two date formats."""
vintages = returns["version"].astype("string").value_counts().sort_index()
print("values in the `version` column:")
print(vintages.to_string())
print()
crosstab = pd.crosstab(returns["year"], returns["version"].astype("string"))
print(crosstab.to_string())
print()
formats = {
    value: "YYYYMMDD" if value.isdigit() else "YYYY-MM-DD"
    for value in vintages.index
}
print(f"date formats in one column: {set(formats.values())}")
print(f"a lexical sort of this column orders them {sorted(vintages.index)}, "
      f"which is not chronological order")


values in the `version` column:
version
20210113      4287
2025-11-20     535

version  20210113  2025-11-20
year                         
1976          332           0
1980          364           0
1984          323           0
1988          285           0
1992          370           0
1996          368           0
2000          369           0
2004          317           0
2008          351           0
2012          316           0
2016          345           0
2020          547           0
2024            0         535

date formats in one column: {'YYYY-MM-DD', 'YYYYMMDD'}
a lexical sort of this column orders them ['20210113', '2025-11-20'], which is not chronological order


The `version` column holds **`20210113`** on 4,287 rows and **`2025-11-20`** on
535 — and the split is exactly by year. Everything through 2020 carries the 2021 vintage; every
2024 row carries the 2025 one.

Two things follow. The first is that a single `md5` and a single dataset version pin the
*file*, not the *data*: the 2024 rows were revised and the earlier ones were not, so
"downloaded once, therefore consistent" is false here.

The second is a parsing hazard. The two values are in **different date formats** —
`YYYYMMDD` and `YYYY-MM-DD` — in one column. A lexical sort orders them
`['20210113', '2025-11-20']`, which happens to be chronologically right by accident and would
not be for a vintage stamped `19991231`. Anything that sorts or compares this column without
parsing it is relying on luck.

**[rule]** Check whether a version or vintage column is constant before treating a download as
a vintage. If it is not, the file is a concatenation and should be described as one.

## 7. What `party_simplified` throws away

The file carries two party columns: `party_detailed` as reported, and `party_simplified`
collapsing everything outside Democrat, Republican and Libertarian into `OTHER`.

In [8]:
"""What `party_simplified` discards."""
other = returns[returns["party_simplified"] == "OTHER"]
print(f"rows labelled OTHER      : {len(other):,} of {len(returns):,}")
print(f"distinct party_detailed  : {other['party_detailed'].nunique()}")
print(f"votes                    : {int(other['candidatevotes'].sum()):,} "
      f"({100 * other['candidatevotes'].sum() / returns['candidatevotes'].sum():.3f}% of all)")
print()
national = other.groupby(["year", "candidate"])["candidatevotes"].sum().reset_index()
largest = national.loc[national.groupby("year")["candidatevotes"].idxmax()]
print("largest candidacy inside OTHER, by year:")
print(largest.sort_values("candidatevotes", ascending=False).head(5).to_string(
    index=False, formatters={"candidatevotes": "{:,.0f}".format}))
print()
print(returns.groupby("year").apply(lambda g: pd.Series({
    "parties": g["party_detailed"].nunique(),
    "OTHER %": 100 * g.loc[g["party_simplified"] == "OTHER", "candidatevotes"].sum()
               / g["candidatevotes"].sum(),
}), include_groups=False).to_string(formatters={
    "parties": "{:,.0f}".format, "OTHER %": "{:.3f}".format}))


rows labelled OTHER      : 2,909 of 4,822
distinct party_detailed  : 207
votes                    : 57,099,629 (3.826% of all)

largest candidacy inside OTHER, by year:
 year         candidate candidatevotes
 1992       PEROT, ROSS     19,722,042
 1996       PEROT, ROSS      7,541,700
 1980 ANDERSON, JOHN B.      5,719,222
 2000      NADER, RALPH      2,542,163
 2016       STEIN, JILL      1,393,155

     parties OTHER %
year                
1976      27   2.395
1980      24   7.522
1984      18   0.875
1988      29   0.893
1992      43  19.755
1996      33   9.986
2000      32   3.856
2004      44   0.991
2008      38   1.488
2012      38   1.163
2016      40   3.138
2020      43   0.786
2024      74   1.571


**2,909 of 4,822 rows are labelled `OTHER`, covering 207 distinct parties and
57,099,629 votes — 3.826% of every vote in the file.**

The largest thing in that bucket is **Ross Perot's 19,722,042 votes in 1992**, which is
**19.755%** of the national vote that year. The second largest is Perot again in 1996 at
7,541,700. John Anderson in 1980, Ralph Nader in 2000 and Jill Stein in 2016 follow.

So in 1992 nearly a fifth of the American presidential vote is, in the simplified column, a
residual category sharing a label with the Prohibition Party. The recode is not wrong — it is
doing exactly what its name says — but a third-party analysis built on `party_simplified` has
thrown away its own subject before it starts.

Note also the last row of that table: **74 distinct parties in 2024**, against 27 in 1976. The
field the recode collapses is the one that has grown most.

## 8. Where the collapse matters, and where it does not

The criticism in section 7 has a limit, and stating it is part of making the criticism
trustworthy.

In [9]:
"""Where the collapse matters, and where it does not."""
leaders = returns.loc[returns.groupby(["year", "state"])["candidatevotes"].idxmax()]
print("party of the leading candidate, by state-year:")
print(leaders["party_simplified"].value_counts().to_string())
print()
print(f"state-years led by a candidate labelled OTHER: "
      f"{int((leaders['party_simplified'] == 'OTHER').sum())} of {len(leaders)}")
print()
denominators = returns.groupby(["year", "state"]).agg(
    published=("totalvotes", "first"), summed=("candidatevotes", "sum")
)
shift = 100 * (denominators["summed"] / denominators["published"] - 1)
print(f"choosing summed votes rather than totalvotes as a denominator moves a share "
      f"by at most {shift.abs().max():.3f} pp")
print(f"  largest: {shift.abs().idxmax()}")
print(f"  median absolute shift: {shift.abs().median():.5f} pp")


party of the leading candidate, by state-year:
party_simplified
REPUBLICAN    389
DEMOCRAT      274

state-years led by a candidate labelled OTHER: 0 of 663

choosing summed votes rather than totalvotes as a denominator moves a share by at most 0.778 pp
  largest: (np.int64(2024), 'DISTRICT OF COLUMBIA')
  median absolute shift: 0.00000 pp


**No state-year in fifty years was led by a candidate labelled `OTHER`** — 389
Republican, 274 Democrat, zero. For the question most people bring to this file, the recode
costs nothing at all, and an analyst who uses `party_simplified` to identify state winners is
not making a mistake.

It is fatal only for the field it collapses. That is the honest shape of the finding: not *this
column is bad*, but *this column answers one question perfectly and destroys another*.

The denominator choice is similarly bounded. Using summed votes rather than the published
`totalvotes` moves a computed share by a **median of 0.00000 pp** and at most **0.778 pp**, in
the District of Columbia in 2024 — the same state whose definition differs. Almost everywhere
the two denominators agree exactly, and the one place they do not is the place section 5 already
identified.

**[rule]** State the bound on a defect as precisely as the defect. "This recode is lossy" and
"this recode changes no state winner in thirteen elections" are both true, and only the pair is
useful.

## Limitations

1. **Zero sampling error is not zero error.** Votes are miscounted, recounted, challenged and
   certified late. This file records certified returns; it does not verify them, and nothing in
   it would show a counting failure.

2. **The identity test bounds internal consistency, not accuracy.** A state whose parts sum to
   its published total can still be wrong in both.

3. **`is_ballot_disposition` is a declared list, not a detector.** Six labels are named in
   `ELECTION_BALLOT_DISPOSITIONS`, five of which appear in this vintage. A new label in a future
   release would not be flagged, and there is no way to tell a person from a disposition
   programmatically.

4. **New York's 874-vote gap is unexplained.** Section 5 shows it is not the disposition rows.
   What it is remains open, and this notebook does not guess.

5. **Presidential returns only, state level.** No congressional, state or local returns; no
   county grain. Turnout is not computed anywhere, because this file carries no denominator
   population.

6. **The `notes` column is 100% null in this vintage.** It exists and carries nothing, so no
   claim here rests on it.

7. **Nothing here measures civic engagement.** Votes cast for president is one narrow indicator
   of one behaviour. The domain name is inherited from the corpus's map, and the notebook is
   about the source rather than the construct.

## Key Takeaways

1. **Sampling error of exactly zero is possible, and it settles less than it sounds.** This is
   the first source in the corpus with no uncertainty column because none could exist.

2. **The identity holds in 661 of 663 state-years as published**, adrift by 3,409 votes.

3. **Cleaning breaks it.** Removing the 48 ballot-disposition rows takes it to 625 of 663 and
   multiplies the discrepancy 139×, because `totalvotes` counts them.

4. **Test an identity before and after cleaning.** A worsening is evidence about your
   assumption, not about the data.

5. **`totalvotes` does not mean one thing.** DC 2024's gap is exactly its disposition rows;
   New York's is not.

6. **One download can be two vintages.** 4,287 rows at `20210113`, 535 at `2025-11-20`, in two
   date formats in one column.

7. **207 parties and 57,099,629 votes are labelled `OTHER`**, including Perot's 19,722,042 in
   1992 — 19.755% of that year's vote.

8. **And the collapse changes no winner in 663 state-years.** Bound the defect as precisely as
   you state it.

9. **Pin against the publisher's checksum, not only your own.** A self-hash detects your copy
   changing; a publisher MD5 detects the file being swapped.

## Exercises

1. **Find New York's 874 votes.** Section 5 rules out the disposition rows. Work through the
   seventeen 2024 New York rows and state what the gap is, or establish that the file cannot
   say.

2. **Test the identity by decade.** Compute the balance rate for 1976–1996 and 2000–2024
   separately, before and after removing dispositions. Does the cleaning trap get worse over
   time, and if so, why would it?

3. **Rebuild the third-party series properly.** Using `party_detailed`, compute each named
   party's national share by year and show at least one conclusion that `party_simplified`
   would have made impossible.

4. **Break the pin.** Change `PINNED_MD5` by one character and run the retrieve cell. Read the
   error. Then say what you would do if it fired for real, in order.

5. **Look for a third meaning.** Section 5 finds two definitions of `totalvotes`. Write a test
   across all 663 state-years that classifies each into "total excludes dispositions", "total
   includes them", or "neither", and report the counts.

6. **Argue for removal.** Construct the analysis for which dropping the disposition rows *is*
   correct, state what you must recompute alongside it, and say how you would report the
   resulting denominator.

## References

- MIT Election Data and Science Lab (2017–). *U.S. President 1976–2024.* Harvard Dataverse,
  V10.1. https://doi.org/10.7910/DVN/42MVDX
- Harvard Dataverse. *Native API — datasets and file access.*
  https://guides.dataverse.org/en/latest/api/native-api.html
- Handbook §1.11 (BEA ACPSA — no sampling error because nothing was sampled), §1.2 (LAUS —
  uncertainty that exists and is not published), §3 (denominators and universes), §5.3 (vintage
  pinning), §15 (this regime).
- `src/krl_tutorials_data/sources.py` (`fetch_dataverse_file`, `fetch_mit_president`) and
  `tests/test_dataverse.py`.